In [8]:
! python -m spacy download en_core_web_sm
! python -m spacy download fr_core_news_sm
! pip install nltk pandas scikit-learn

     ---------------------------------------- 0.0/12.8 MB ? eta -:--:--
      --------------------------------------- 0.3/12.8 MB ? eta -:--:--
     - -------------------------------------- 0.5/12.8 MB 1.7 MB/s eta 0:00:08
     -- ------------------------------------- 0.8/12.8 MB 2.0 MB/s eta 0:00:07
     --- ------------------------------------ 1.0/12.8 MB 1.4 MB/s eta 0:00:09
     ----- ---------------------------------- 1.8/12.8 MB 1.8 MB/s eta 0:00:06
     ----- ---------------------------------- 1.8/12.8 MB 1.8 MB/s eta 0:00:06
     ------ --------------------------------- 2.1/12.8 MB 1.7 MB/s eta 0:00:07
     ------- -------------------------------- 2.4/12.8 MB 1.6 MB/s eta 0:00:07
     -------- ------------------------------- 2.6/12.8 MB 1.5 MB/s eta 0:00:07
     -------- ------------------------------- 2.6/12.8 MB 1.5 MB/s eta 0:00:07
     --------- ------------------------------ 2.9/12.8 MB 1.3 MB/s eta 0:00:08
     --------- ------------------------------ 2.9/12.8 MB 1.3 MB/s

# Exercise 1 : Lemmatization

In this exercise, the objective is to create your own lemmatizer for french language. We will test different lemmatization approaches : 
* Based on a dictionary
* Based on machine learning approach (you can use sklearn) or define your own architecture with pytorch
* With and without pos tag given as input

In all case you should compare your results and report performances of the proposed algorithm to [spacy](https://spacy.io/models/fr) lemmatizer (the different configuration).

You are free to use any machine-learning algorihtm/model, taking or not the context of sentences such as [LinearRegression](https://scikit-learn.org/1.5/modules/generated/sklearn.linear_model.LinearRegression.html) or training your own [RNN with pytorch](https://pytorch.org/docs/stable/generated/torch.nn.RNN.html). 
However you must always motivate your choices and compare results of the different configurations.

You will send the report to *thomas.gerald@universite-paris-saclay.fr* in PDF format named as following and the code (notebook with  output of the two exercises in a zip format) :


**report_[firstname]_[lastname].pdf**

The report for the two exercises must not exceed three pages !


## Dataset
To train or build your lemmatizer you have three files in *tabular separated values* format :
* [training-set.tsv](https://thomas-gerald.fr/TMC/resources/data/training-set.tsv) that you can use to train/build your dictionnary/model 
* [testing-set.tsv](https://thomas-gerald.fr/TMC/resources/data/testing-set.tsv) used to evaluate the different approaches
* [testing-gallica.tsv](https://thomas-gerald.fr/TMC/resources/data/testing-gallica.tsv) used as gold standard to evaluate performances [github (in french)](https://github.com/Gallicorpora/Lemmatisation)

In our case we have two possibilities for a lemma:
* (a) A sequence of characters, meaning that "to rule" an "a rule" are the same lemma
* (b) A sequence of characters, meaning that "to rule" represent the verb, a tuple ("rule", "V") while "a rule" is represented by the tuple ("rule", "N") 
In the (a) case the size of the vocabulary (output) will be 
## Spacy :

Below a small example using spacy lemmatization
```python
import spacy
nlp = spacy.load("en_core_web_sm")
text_a = "He is thirty years old"
text_b = "We still are champions"
print(f'Lemmatization A : {[(w.lemma_, w.pos_) for w in nlp(text_a)]}')
print(f'Lemmatization B : {[(w.lemma_, w.pos_) for w in nlp(text_b)]}')
```

### Reading data

You can use pandas to read the data using tabular separator as following

In [9]:
import pandas as pd
train_file = "data/training-set.tsv"
pd.read_csv(train_file, sep='\t', names=["token", "lemma", "pos"])

,token,lemma,pos
0,Certes,certes,ADV
1,",",",",PONCT
2,rien,rien,PRO
3,ne,ne,ADV
4,dit,dire,V
...,...,...,...
261384,effet,effet,N
261385,positif,positif,A
261386,.,.,PONCT
261387,tenir,tenir,V


In [10]:
w2lemma = {}

w_vocabulary = {}
l_vocabulary = set()
lp_vocabulary = set()

with open(train_file, 'r')  as f:
    for line in f:
        try: 
            word, lemma, pos = line.split()
            w2lemma[word] = lemma
            # l_vocabulary.add(lemma)
            # lp_vocabulary.add((lemma, pos))
        except: 
            pass

print(f'The input vocabulary contains : {len(w2lemma)} words' )
# print(f'The number of str lemma is :  {len(l_vocabulary)}')
# print(f'The number of lemma (considering PoS) is :  {len(lp_vocabulary)}')

The input vocabulary contains : 23270 words


Now we test it

In [ ]:
def lemmatizer_dict(word, dictionary):
    if word not in dictionary.keys():
        return "Not found"
    lemma = dictionary[word]
    return lemma

test_file = "data/testing-set.tsv"
with open(test_file, 'r')  as f:
    for line in f:
        if len(line.split()) == 3: # some lines are empty
            word, lemma, pos = line.split()
        print(f"'{word}' has lemma '{lemmatizer_dict(word, w2lemma)}'")

'Non' has lemma 'non'
'compensée' has lemma 'compenser'
'salarialement' has lemma 'Not found'
'(' has lemma '('
'le_plus_souvent' has lemma 'le_plus_souvent'
')' has lemma ')'
',' has lemma ','
'elle' has lemma 'il'
'provoque' has lemma 'provoquer'
'une' has lemma 'un'
'augmentation' has lemma 'augmentation'
'de' has lemma 'de'
'la' has lemma 'le'
'productivité' has lemma 'productivité'
'du' has lemma 'de'
'travail' has lemma 'travail'
'qui' has lemma 'qui'
'permet' has lemma 'permettre'
'de' has lemma 'de'
'financer' has lemma 'financer'
'son' has lemma 'son'
'coût' has lemma 'coût'
'et' has lemma 'et'
'de' has lemma 'de'
'l'' has lemma 'le'
'absorber' has lemma 'absorber'
'sans_qu'' has lemma 'sans_que'
'il' has lemma 'il'
'y' has lemma 'y'
'paraisse' has lemma 'Not found'
'davantage' has lemma 'davantage'
'.' has lemma '.'
'.' has lemma '.'
'Les' has lemma 'le'
'conséquences' has lemma 'conséquence'
'sur' has lemma 'sur'
'l'' has lemma 'le'
'embauche' has lemma 'embauche'
'sont' has

Evaluate the accuracy. Those that are not found are counted as missed.

In [16]:
correct_lemmas = 0
total = 0
test_file = "data/testing-set.tsv"
with open(test_file, 'r')  as f:
    for line in f:
        if len(line.split()) == 3: # some lines are empty
            word, lemma, pos = line.split()
        retrieved_lemma = lemmatizer_dict(word, w2lemma)
        print(f"'{word}' has lemma '{retrieved_lemma}'")
        if retrieved_lemma == lemma:
            correct_lemmas += 1
        total += 1
print(f"Total accuracy is {correct_lemmas / total}")

'Non' has lemma 'non'
'compensée' has lemma 'compenser'
'salarialement' has lemma 'Not found'
'(' has lemma '('
'le_plus_souvent' has lemma 'le_plus_souvent'
')' has lemma ')'
',' has lemma ','
'elle' has lemma 'il'
'provoque' has lemma 'provoquer'
'une' has lemma 'un'
'augmentation' has lemma 'augmentation'
'de' has lemma 'de'
'la' has lemma 'le'
'productivité' has lemma 'productivité'
'du' has lemma 'de'
'travail' has lemma 'travail'
'qui' has lemma 'qui'
'permet' has lemma 'permettre'
'de' has lemma 'de'
'financer' has lemma 'financer'
'son' has lemma 'son'
'coût' has lemma 'coût'
'et' has lemma 'et'
'de' has lemma 'de'
'l'' has lemma 'le'
'absorber' has lemma 'absorber'
'sans_qu'' has lemma 'sans_que'
'il' has lemma 'il'
'y' has lemma 'y'
'paraisse' has lemma 'Not found'
'davantage' has lemma 'davantage'
'.' has lemma '.'
'.' has lemma '.'
'Les' has lemma 'le'
'conséquences' has lemma 'conséquence'
'sur' has lemma 'sur'
'l'' has lemma 'le'
'embauche' has lemma 'embauche'
'sont' has

We think this is a good accuracy given that there are 23270 words in total in the vocabulary, some of which are repeated. And french has between 60,000 and 100,000 words in total.